# Project Gutenberg Data Collection

Target page: https://www.gutenberg.org/browse/languages/tl

Reference code: https://shravan-kuchkula.github.io/scrape_clean_normalize_gutenberg_text/

In [1]:
import os
import pandas as pd
import glob
import json

In [2]:
temp_path = os.getcwd().split('\\')
project_dir = '\\'.join(temp_path[:temp_path.index('SOURCE') + 1])

## Setup

In [3]:
########################################
#  Module: gutenbergPreprocessing.py
#  Author: Shravan Kuchkula
#  Date: 05/24/2019
########################################

import re
import nltk
import string
import requests
from bs4 import BeautifulSoup

def remove_gutenburg_headers(book_text):
    book_text = book_text.replace('\r', '')
    book_text = book_text.replace('\n', ' ')
    start_match = re.search(r'\*{3}\s?START.+?\*{3}', book_text)
    end_match = re.search(r'\*{3}\s?END.+?\*{3}', book_text)
    try:
        book_text = book_text[start_match.span()[1]:end_match.span()[0]]
    except AttributeError:
        print('No match found')    
    return book_text

def remove_gutenberg_footer(book_text):
    if book_text.find('End of the Project Gutenberg') != -1:
        book_text = book_text[:book_text.find('End of the Project Gutenberg')]
    elif book_text.find('End of Project Gutenberg') != -1:
        book_text = book_text[:book_text.find('End of Project Gutenberg')]
    return book_text

def getTextFromURLByRemovingHeaders(book_urls):
    book_texts = []
    for url in book_urls:
        book_text = requests.get(url).text
        book_title = get_title(book_text)
        book_text = remove_gutenburg_headers(book_text)
        book_text = remove_gutenberg_footer(book_text)
        book_texts.append([book_title, book_text])
    return book_texts

In [4]:
def get_page(URL):
    headers = {
        'User-Agent': "Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.90 Mobile Safari/537.36"
    }

    page = requests.get(URL, headers=headers)
    return [BeautifulSoup(page.content.decode(), 'html.parser'), page.status_code]

def get_urls_to_plaintext(books_page):
    lst = books_page[0].find("div", class_="pgdbbylanguage").find_all("li", class_="pgdbetext")
    links = [li.find("a")["href"] for li in lst]
    book_id = [link.split('/')[2] for link in links]
    links_to_plaintext = [f"https://www.gutenberg.org/cache/epub/{id}/pg{id}.txt" for id in book_id]
    return links_to_plaintext

def get_title(text):
    start = text.find('Title:')
    end = text.find('Author:', start)
    output = text[start:end].replace('\n', '').replace('Title: ', '').replace('\r', '').replace('$', '')

    return output

## Collect Data

In [ ]:
tagalog_books_page = get_page("https://www.gutenberg.org/browse/languages/tl")
tagalog_books_page

In [ ]:
book_urls = get_urls_to_plaintext(tagalog_books_page)
book_urls

In [ ]:
# Get book text and title (headers & footers removed)
%time book_texts = getTextFromURLByRemovingHeaders(book_urls)

In [ ]:
book_texts[10:15]

In [ ]:
# Total books
len(book_texts)

## Save to text file

In [10]:
path = f"{project_dir}/data/1 - Data Collection/books/gutenberg/"

if not os.path.exists(path):
  os.makedirs(path)

In [11]:
def save_txtfile(filename, content):
    f = open(path + filename, "w", encoding="utf-8")
    f.write(content)
    f.close()
    print(f"Saved {filename}")

In [ ]:
for item in book_texts:
    title = item[0]
    filename = title + '.txt'
    content = item[1]
    save_txtfile(filename.replace('"', ""), content)

In [ ]:
# Check directory
!dir